# CellCommunication / C2C

Package-backed ChronoPack temporal cell-cell communication workflow. The notebook keeps dataset paths, biological choices, inspection, and manuscript-specific callouts.

In [ ]:
from pathlib import Path
import os

import scanpy as sc
import pandas as pd

import chronopack as cp

sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5, 5))


In [ ]:
WORKSPACE_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = WORKSPACE_DIR if WORKSPACE_DIR.name == "c2c" else WORKSPACE_DIR / "c2c"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def original_project_dir() -> Path:
    for marker_name in ("original_dir.txt", "OriginalDirectoryPath.txt"):
        marker = NOTEBOOK_DIR / marker_name
        if marker.exists():
            raw = Path(os.path.expanduser(marker.read_text().strip()))
            return raw.parent.parent if raw.suffix == ".ipynb" or raw.is_file() else raw
    return Path("/new-stg/home/alex/projects/chrono_seq/analyses/thp1-k562-lps.hg38.rep")

PROJECT_DIR = original_project_dir()
REPRO_INPUT_DIR = NOTEBOOK_DIR / "reproduction" / "input_files"

def input_file(name: str, fallback: Path) -> Path:
    local = REPRO_INPUT_DIR / name
    return local if local.exists() else fallback

CELLCHAT_PATH = input_file(
    "interaction_input_CellChatDB.csv",
    Path("/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv"),
)
ADATA_PATH = input_file(
    "adata_full.chk1.h5ad",
    PROJECT_DIR / "output" / "adata_full.chk1.h5ad",
)
ASSO_BOTH_PATH = input_file(
    "asso_df.both.2trajectory.nknots4.expanded.csv",
    PROJECT_DIR / "intmd" / "sc_to_tradeSeq" / "asso_df.both.2trajectory.nknots4.expanded.csv",
)
TS_EXPORT_GRID_DIR = PROJECT_DIR / "intmd" / "sc_to_tradeSeq" / "ts_export_grid_100pts"

SENDER_CELL_TYPE = "THP1"
RECEIVER_CELL_TYPE = "K562"
CELL_TYPE_KEY = "cell_type"
COUNTS_LAYER = "counts"
EXPR_LAYER = "log1p"
MIN_DETECT = 1e-100
CAUSAL_WINDOW_MIN = (0, 45)
DOWNSAMPLE_INTERVAL_MIN = 15
PREFER_FITS_FOR_SCORING = False
RESPONSE_GENES = ["NFKBIA", "NFKBIZ", "TNFAIP3", "BIRC3", "IER3", "CXCL1", "CXCL2", "CXCL3"]

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ADATA_PATH: {ADATA_PATH}")
print(f"ASSO_BOTH_PATH: {ASSO_BOTH_PATH}")
print(f"TS_EXPORT_GRID_DIR: {TS_EXPORT_GRID_DIR}")


In [ ]:
adata_full = sc.read_h5ad(ADATA_PATH)
print(f"AnnData full: {adata_full.shape}")


In [ ]:
c2c_result = cp.tl.c2c.infer(
    adata=adata_full,
    lr_resource=CELLCHAT_PATH,
    association=ASSO_BOTH_PATH,
    sender=SENDER_CELL_TYPE,
    receiver=RECEIVER_CELL_TYPE,
    response_genes=RESPONSE_GENES,
    cell_type_key=CELL_TYPE_KEY,
    counts_layer=COUNTS_LAYER,
    expression_layer=EXPR_LAYER,
    min_detect=MIN_DETECT,
    causal_window_min=CAUSAL_WINDOW_MIN,
    association_p_col="pvalue_2",
    prefer_fits=PREFER_FITS_FOR_SCORING,
    tradeseq_export_dir=TS_EXPORT_GRID_DIR,
    provenance={"workflow": "c2c/CellCommunication.ipynb"},
)

print(f"Candidate LR pairs: {c2c_result.candidate_interactions.shape}")
print(f"Scored LR pairs: {c2c_result.interactions.shape}")
print(f"Scored ligands: {c2c_result.ligand_scores.shape}")
c2c_result.ligand_scores.head(20)


In [ ]:
pair_scores_out = c2c_result.legacy_pair_scores(interval_min=DOWNSAMPLE_INTERVAL_MIN, include_rank=False)
pair_scores_out["logp"] = pair_scores_out["logp"].round(12)
pair_scores_ranked = c2c_result.legacy_pair_scores(include_rank=True)

pair_scores_out.to_csv(OUTPUT_DIR / "chronopack_lr_pair_scores.csv", index=False)
pair_scores_ranked.to_csv(OUTPUT_DIR / "chronopack_lr_pair_scores.with_rank.csv", index=False)
c2c_result.ligand_scores.to_csv(OUTPUT_DIR / "chronopack_ligand_scores.csv", index=False)
c2c_result.write(OUTPUT_DIR / "chronopack_c2c", prefix="chronopack_c2c")


In [ ]:
fig, ax = cp.pl.c2c.interaction_scatter(
    c2c_result,
    receptor_detection_clip=0.2,
    p_thresh=0.05,
    x_thresh=0.1,
    max_labels=20,
    title="ChronoPack LR pair prioritization",
    xlabel="early LR activity (0-45 min)",
    ylabel="ligand -log10(p)",
)
fig.savefig(OUTPUT_DIR / "chronopack_lr_pair_scores.png", dpi=300, bbox_inches="tight")


In [ ]:
adata_thp1, adata_k562 = cp.tl.c2c.split_by_cell_type(
    adata_full,
    cell_type_key=CELL_TYPE_KEY,
    sender=SENDER_CELL_TYPE,
    receiver=RECEIVER_CELL_TYPE,
)
tradeseq_exports = cp.tl.c2c.load_tradeseq_grid_exports(TS_EXPORT_GRID_DIR)
tnf_pairs = pd.DataFrame({"ligand": ["TNF", "TNF"], "receptor": ["TNFRSF1A", "TNFRSF1B"]})
tnf_pair_series = cp.tl.c2c.build_pair_series(
    tnf_pairs,
    sender_adata=adata_thp1,
    receiver_adata=adata_k562,
    response_genes=["NFKBIA"],
    time_key="time",
    layer=EXPR_LAYER,
    prefer_fits=True,
    fits_df=tradeseq_exports["fits"],
    wmeans_df=tradeseq_exports["wmeans"],
)

if ("TNF", "TNFRSF1A") in tnf_pair_series:
    fig, ax = cp.tl.c2c.plot_pair_overlay(
        tnf_pair_series,
        pair=("TNF", "TNFRSF1A"),
        norm_mode="minmax",
        nfkb_onset_time=70,
        figsize=(4, 2.5),
        title="TNF->TNFRSF1A and K562 NFKBIA",
    )
    fig.savefig(OUTPUT_DIR / "TNF_TNFRSF1A_NFKBIA_overlay.png", dpi=300, bbox_inches="tight")
else:
    print("TNF->TNFRSF1A was not available for overlay plotting.")


In [ ]:
genes = ["TNF", "TNFRSF1A", "TNFRSF1B", "NFKBIA", "TNFAIP3", "BIRC3"]
genes = [gene for gene in genes if gene in adata_full.var_names]
fig, axes = cp.pl.c2c.plot_genes_over_time(
    adata_full,
    genes,
    time_key="time",
    cluster_key=CELL_TYPE_KEY,
    layer=EXPR_LAYER,
    ncols=3,
)
fig.savefig(OUTPUT_DIR / "selected_gene_expression_over_time.png", dpi=300, bbox_inches="tight")
